## Network Structure Posteriors — example5

Reads the executed `examples/example5.ipynb` and extracts the posterior probability
that the **correct** network structure was identified in each stoichiometric channel.

Each channel's MCMC independently selects which candidate reactions are 'on'.
The bitstring printed by `calc_network_posteriors` (e.g. `01010`) encodes which
reactions are active; the probability shown is the posterior mass on that structure.

Run this notebook from the `writing/` directory.

In [1]:
import json
import re
import sys
import os
import pandas as pd

# Path to the executed example5 notebook
NOTEBOOK_PATH = os.path.join('..', 'examples', 'example5.ipynb')

# Cell indices of the three MCMC blocks in example5.ipynb
# (0-indexed; update if cells are added/removed)
MCMC_CELL_INDICES = {
    'T=100': 8,
    'T=200': 11,
    'T=400': 15,
}

with open(NOTEBOOK_PATH) as f:
    nb = json.load(f)

print(f'Loaded {NOTEBOOK_PATH}')
print(f'Total cells: {len(nb["cells"])}')
for T, idx in MCMC_CELL_INDICES.items():
    cell = nb['cells'][idx]
    n_out = len(cell.get('outputs', []))
    exec_n = cell.get('execution_count')
    print(f'  {T} → cell {idx}: execution_count={exec_n}, outputs={n_out}')


Loaded ../examples/example5.ipynb
Total cells: 17
  T=100 → cell 8: execution_count=8, outputs=15
  T=200 → cell 11: execution_count=10, outputs=17
  T=400 → cell 15: execution_count=13, outputs=17


In [2]:
def extract_network_posteriors(cell):
    """
    Parse all stream output from an MCMC cell.
    Returns a dict: {channel_index: (count, top_structure, top_prob)}
    where top_structure is the bitstring with the highest posterior probability.
    """
    full_text = ''
    for out in cell.get('outputs', []):
        if out.get('output_type') == 'stream':
            full_text += ''.join(out.get('text', []))

    results = {}
    blocks = re.split(r'(?=\nProcessing Index)', '\n' + full_text)

    for block in blocks:
        idx_match   = re.search(r'Processing Index: (\d+)', block)
        count_match = re.search(r'Total Count = (\d+)', block)
        if not idx_match:
            continue

        ch_idx = int(idx_match.group(1))
        count  = int(count_match.group(1)) if count_match else 0

        # Find all structure lines: e.g. '01010 (R2 + R4) : 0.9987'
        struct_hits = re.findall(
            r'([01]+)\s*\(([^)]+)\)\s*:\s*([0-9.]+)',
            block
        )

        if struct_hits:
            # Pick the one with highest probability
            best = max(struct_hits, key=lambda x: float(x[2]))
            results[ch_idx] = {
                'count':     count,
                'bitstring': best[0],
                'reactions': best[1].strip(),
                'prob':      float(best[2]),
            }
        elif count > 0 and 'No network structure' not in block:
            # Channel ran but only 1 candidate reaction — no network comparison
            results[ch_idx] = {
                'count':     count,
                'bitstring': None,
                'reactions': '(single reaction)',
                'prob':      None,
            }

    return results


# Extract posteriors for each T
all_results = {}
for T, idx in MCMC_CELL_INDICES.items():
    all_results[T] = extract_network_posteriors(nb['cells'][idx])
    print(f'{T}: found {len(all_results[T])} channels with data')


T=100: found 15 channels with data
T=200: found 15 channels with data
T=400: found 15 channels with data


In [3]:
# ── Build the summary table ──────────────────────────────────────────────────
#
# For channels where calc_network_posteriors ran (>=2 candidate reactions),
# we show the top structure and its posterior probability.
# For single-reaction channels, the correct structure is trivially identified
# (the reaction is always 'on' if the channel fired at all).

T_values = ['T=100', 'T=200', 'T=400']

# Collect all channel indices that appear in any T
all_channels = sorted(
    set(ch for res in all_results.values() for ch in res)
)

rows = []
for ch in all_channels:
    # Use T=100 as reference for correct structure (same network across all T)
    ref = None
    for T in T_values:
        if ch in all_results[T]:
            ref = all_results[T][ch]
            break

    correct_structure = (
        f"{ref['bitstring']} ({ref['reactions']})"
        if ref and ref['bitstring']
        else ref['reactions'] if ref else '—'
    )

    row = {'Channel': ch, 'Correct Structure': correct_structure}

    for T in T_values:
        data = all_results[T].get(ch)
        if data is None:
            row[f'{T} Count'] = '—'
            row[f'{T} Posterior'] = '—'
        elif data['prob'] is None:
            row[f'{T} Count'] = data['count']
            row[f'{T} Posterior'] = 'single rxn'
        else:
            row[f'{T} Count']     = data['count']
            row[f'{T} Posterior'] = f"{data['prob']:.4f}"

    rows.append(row)

df = pd.DataFrame(rows)
df = df.set_index('Channel')

# Pretty column order
cols = ['Correct Structure']
for T in T_values:
    cols += [f'{T} Count', f'{T} Posterior']
df = df[cols]

print('Network Structure Posteriors — example5')
print('Posterior = P(correct network | data), extracted from executed example5.ipynb')
print()
display(df)


Network Structure Posteriors — example5
Posterior = P(correct network | data), extracted from executed example5.ipynb



,Correct Structure,T=100 Count,T=100 Posterior,T=200 Count,T=200 Posterior,T=400 Count,T=400 Posterior
Channel,,,,,,,
0,01000 (R2),255,1.0000,512,1.0000,1054,1.0000
1,01010 (R2 + R4),378,0.9987,713,1.0000,1400,0.9998
3,01001 (R2 + R5),184,0.9996,398,0.9998,766,1.0000
5,(single reaction),64,single rxn,127,single rxn,252,single rxn
25,10000 (R1),580,0.9969,1143,0.9993,2260,0.9927
36,01100 (R2 + R3),427,0.9998,860,0.9998,1717,1.0000
37,00001 (R5),138,1.0000,252,1.0000,482,1.0000
43,(single reaction),317,single rxn,603,single rxn,1197,single rxn
44,00101 (R3 + R5),310,0.9998,626,1.0000,1232,0.9998


### Notes

- **Correct Structure**: the bitstring identifies which of the candidate reactions in that
  channel are active. E.g. `01010 (R2 + R4)` means reactions 2 and 4 are on, 1/3/5 are off.
- **Posterior**: fraction of post-burn-in MCMC samples in which exactly the correct
  set of reactions was identified as active (above epsilon threshold).
- **Single rxn**: only one candidate reaction in that channel — network structure is
  trivially determined by whether the channel fired at all.
- Channels not listed here had zero observed jumps and were skipped by the MCMC.
- The Bayes factor between the correct model and the next-best model is
  approximately `p_correct / p_other`; since `p_correct ≈ 1.0` and the remaining
  probability is spread across all other structures, BFs are typically >>1000.